# Kronos Stock Predictor — 02 训练可视化

训练 Tokenizer + Predictor 并可视化损失曲线

In [ ]:
import sys; sys.path.insert(0, '..')
import torch, numpy as np, matplotlib.pyplot as plt
from config.default_config import Config
from config.model_configs import build_tokenizer_config, build_model_config
from data.dataset import StockDataset
from model.kronos_tokenizer import KronosTokenizer
from model.kronos_model import Kronos
from train.losses import tokenizer_loss, predictor_loss
from train.training_utils import set_seed
from torch.utils.data import DataLoader

set_seed(42)
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
config = Config()
config.dataset_path = './data/processed'
config.batch_size = 16
config.epochs = 3

train_ds = StockDataset(config.dataset_path, config, 'train')
val_ds = StockDataset(config.dataset_path, config, 'val')
train_loader = DataLoader(train_ds, batch_size=config.batch_size, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=config.batch_size, shuffle=False, drop_last=False)
print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')

In [ ]:
# Train Tokenizer
tok_cfg = build_tokenizer_config('mini')
tokenizer = KronosTokenizer(**tok_cfg).to(device)
optimizer = torch.optim.AdamW(tokenizer.parameters(), lr=2e-4)

tok_losses = []
for epoch in range(3):
    tokenizer.train()
    for batch_x, _ in train_loader:
        batch_x = batch_x.to(device)
        optimizer.zero_grad()
        (z_pre, z_full), bsq_loss, _, _ = tokenizer(batch_x)
        loss = tokenizer_loss(batch_x, z_pre, z_full, bsq_loss)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(tokenizer.parameters(), 5.0)
        optimizer.step()
        tok_losses.append(loss.item())
    print(f'Epoch {epoch+1}: avg loss = {np.mean(tok_losses[-len(train_loader):]):.4f}')

plt.plot(tok_losses, alpha=0.3, label='batch')
plt.plot(np.convolve(tok_losses, np.ones(50)/50, mode='valid'), label='smoothed')
plt.xlabel('Batch'); plt.ylabel('Loss'); plt.title('Tokenizer Training Loss'); plt.legend(); plt.show()

In [ ]:
# Train Predictor
model_cfg = build_model_config('mini')
model = Kronos(**model_cfg).to(device)
tokenizer.eval()
for p in tokenizer.parameters():
    p.requires_grad = False

optimizer = torch.optim.AdamW(model.parameters(), lr=4e-5)
pred_losses = []
for epoch in range(3):
    model.train()
    for batch_x, batch_stamp in train_loader:
        batch_x = batch_x.to(device)
        batch_stamp = batch_stamp.to(device)
        with torch.no_grad():
            s1_ids, s2_ids = tokenizer.encode(batch_x, half=True)
        input_s1, input_s2 = s1_ids[:,:-1], s2_ids[:,:-1]
        target_s1, target_s2 = s1_ids[:,1:], s2_ids[:,1:]
        optimizer.zero_grad()
        s1_logits, s2_logits = model(input_s1, input_s2, stamp=batch_stamp[:,:-1,:])
        loss, _, _ = predictor_loss(s1_logits, s2_logits, target_s1, target_s2)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        pred_losses.append(loss.item())
    print(f'Epoch {epoch+1}: avg loss = {np.mean(pred_losses[-len(train_loader):]):.4f}')

plt.plot(pred_losses, alpha=0.3, label='batch')
plt.plot(np.convolve(pred_losses, np.ones(50)/50, mode='valid'), label='smoothed')
plt.xlabel('Batch'); plt.ylabel('Loss'); plt.title('Predictor Training Loss'); plt.legend(); plt.show()